# MacroHFT: results used in the review

Six frozen ETH experts; 0.2 ETH per long position; 0.02% transaction fee.
Run all cells in order. The first run downloads about 455 MB and computes the expert outputs; later runs reuse `data/`.
The test results are exploratory reuse of the published sample.

[Review](presentation/main.pdf) · [Original implementation](https://github.com/ZONG0004/MacroHFT/tree/31e5ef41f93b2aea6e63e6ae2267675c997a8e54)

In [2]:
%pip install -q -r requirements.txt

In [3]:
from pathlib import Path
from urllib.request import urlretrieve
import runpy

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)
DATA = Path("data")
UPSTREAM = DATA / "upstream"
COMMIT = "31e5ef41f93b2aea6e63e6ae2267675c997a8e54"
SOURCE = f"https://raw.githubusercontent.com/ZONG0004/MacroHFT/{COMMIT}"
EXPERTS = [f"{family}/{i}" for family in ("slope", "vol") for i in (1, 2, 3)]
HOLDING, FEE = 0.2, 0.0002
MINUTES_PER_YEAR = 365 * 24 * 60

## Download inputs

In [4]:
def download(url, path):
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {path.name}", flush=True)
    _, headers = urlretrieve(url, partial)
    if headers.get_content_type() == "text/html":
        raise RuntimeError(f"The data host returned a web page instead of {path.name}")
    partial.replace(path)


DATA.mkdir(exist_ok=True)
file_ids = {
    "train": "1iHUs5726pFGYzOwO62JQhNoOu0MvZdT4",
    "val": "1T-O4qaidU-PUpW3XC9lT11bzdGhPqS3R",
    "test": "1FGPpH7j4-0dLGuR4TRiJkiwE_gHJxFEA",
}
for split, file_id in file_ids.items():
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    download(url, DATA / f"df_{split}.feather")

for filename, source_path in {
    "net.py": "model/net.py",
    "decomposition.py": "preprocess/decomposition.py",
    "single_features.npy": "data/feature_list/single_features.npy",
    "trend_features.npy": "data/feature_list/trend_features.npy",
}.items():
    download(f"{SOURCE}/{source_path}", UPSTREAM / filename)

for expert in EXPERTS:
    source_path = f"result/low_level/ETHUSDT/best_model/{expert}/best_model.pkl"
    download(f"{SOURCE}/{source_path}", UPSTREAM / f"{expert.replace('/', '_')}.pkl")

In [5]:
preprocessing = runpy.run_path(str(UPSTREAM / "decomposition.py"))
single_features = np.load(UPSTREAM / "single_features.npy", allow_pickle=True).tolist()
trend_features = np.load(UPSTREAM / "trend_features.npy", allow_pickle=True).tolist()
features = single_features + trend_features

raw_prices = {
    split: pd.read_feather(DATA / f"df_{split}.feather", columns=["timestamp", "close"])
    for split in ("train", "test")
}
frames = {}
for split in ("val", "test"):
    path = DATA / f"{split}_prepared.feather"
    if not path.exists():
        print(f"Preparing {split}: original 360-minute rolling indicators", flush=True)
        raw = pd.read_feather(DATA / f"df_{split}.feather")
        frame = preprocessing["label_whole"](raw).dropna().iloc[1:].reset_index(drop=True)
        frame = frame[["timestamp", "close", *features, "slope_360", "vol_360"]]
        frame.to_feather(path)
        del raw, frame
    frames[split] = pd.read_feather(path)
    frame = frames[split]
    assert frame.timestamp.is_unique and frame.timestamp.is_monotonic_increasing
    assert frame.timestamp.diff().dropna().eq(pd.Timedelta(minutes=1)).all()
    assert np.isfinite(frame[features + ["close", "slope_360", "vol_360"]]).all().all()

assert frames["val"].timestamp.iloc[-1] < frames["test"].timestamp.iloc[0]
display(pd.DataFrame({
    split: {"First": frame.timestamp.iloc[0], "Last": frame.timestamp.iloc[-1],
            "Rows": len(frame)}
    for split, frame in frames.items()
}).T)

,First,Last,Rows
val,2023-02-01 06:00:00,2023-05-31 23:59:00,172440
test,2023-06-01 06:00:00,2023-10-31 23:59:00,219960


## Frozen expert outputs

Both possible previous positions are evaluated. Each strategy then follows its own position sequence.

In [6]:
SubAgent = runpy.run_path(str(UPSTREAM / "net.py"))["subagent"]
models = []
for expert in EXPERTS:
    weights = torch.load(
        UPSTREAM / f"{expert.replace('/', '_')}.pkl", map_location="cpu", weights_only=True
    )
    model = SubAgent(len(single_features), len(trend_features), 2, weights["fc1.weight"].shape[0])
    model.load_state_dict(weights)
    models.append(model.eval())


def expert_advantages(split):
    frame = frames[split]
    n = len(frame) - 1
    path = DATA / f"{split}_advantages.npy"
    if path.exists():
        values = np.load(path)
        assert values.shape == (n, 2, 6) and np.isfinite(values).all()
        return values

    single = torch.tensor(frame[single_features].to_numpy(), dtype=torch.float32)
    trend = torch.tensor(frame[trend_features].to_numpy(), dtype=torch.float32)
    values = np.empty((n, 2, 6), dtype=np.float32)
    # Batch size one preserves the original network's floating-point calculations.
    with torch.inference_mode():
        for i, model in enumerate(models):
            for previous in (0, 1):
                position = torch.tensor([previous], dtype=torch.long)
                for t in range(n):
                    q = model(single[t:t + 1], trend[t:t + 1], position)[0]
                    values[t, previous, i] = (q[1] - q[0]).item()
            print(f"{split}: {EXPERTS[i]} complete", flush=True)
    assert np.isfinite(values).all()
    np.save(path, values)
    return values


d_val = expert_advantages("val")

## Trading and accounting

Positions start flat. Final open positions are marked to market. MTM metrics use the same initial capital; author-style return uses the strategy's `required_money`.

In [7]:
def actions_from_scores(scores, keep_ties=False, cost=0.0):
    actions = np.empty(len(scores), dtype=np.int8)
    previous = 0
    for t, values in enumerate(scores):
        score = values[previous] - cost * (1 - 2 * previous)
        action = previous if keep_ties and score == 0 else int(score > 0)
        actions[t] = action
        previous = action
    return actions


def account(frame, actions):
    price = frame.close.to_numpy(dtype=np.float64)
    actions = np.asarray(actions, dtype=np.int8)
    assert len(actions) == len(price) - 1 and np.isin(actions, [0, 1]).all()
    changes = HOLDING * np.diff(np.r_[0, actions])
    fees = FEE * np.abs(changes) * price[:-1]
    step_pnl = HOLDING * actions * np.diff(price) - fees
    pnl = np.r_[0.0, np.cumsum(step_pnl)]
    capital = HOLDING * price[0] * (1 + FEE)
    equity = capital + pnl
    cash = np.cumsum(-changes * price[:-1] - fees)
    required_money = -cash.min()
    final_balance = cash[-1] + HOLDING * actions[-1] * price[-1]
    np.testing.assert_allclose(pnl[1:], cash + HOLDING * actions * price[1:], atol=1e-8)
    assert (equity > 0).all()
    returns = np.diff(equity) / equity[:-1]
    volatility = returns.std(ddof=0)
    metrics = {
        "MTM Return (%)": 100 * pnl[-1] / capital,
        "Author-style Return (%)": 100 * final_balance / required_money if required_money > 0 else np.nan,
        "Sharpe": returns.mean() / volatility * np.sqrt(MINUTES_PER_YEAR) if volatility > 0 else np.nan,
        "MDD (%)": 100 * (1 - equity / np.maximum.accumulate(equity)).max(),
        "Exposure (%)": 100 * actions.mean(),
    }
    return {"actions": actions, "fees": fees, "step_pnl": step_pnl, "metrics": metrics}


def evaluate(frame, scores, **kwargs):
    return account(frame, actions_from_scores(scores, **kwargs))


validation = {
    expert: evaluate(frames["val"], d_val[:, :, i])
    for i, expert in enumerate(EXPERTS)
}
validation_metrics = pd.DataFrame({name: result["metrics"] for name, result in validation.items()}).T
best_return = validation_metrics["MTM Return (%)"].idxmax()
best_sharpe = validation_metrics["Sharpe"].idxmax()
display(pd.Series({
    "Selected by validation return": best_return,
    "Selected by validation Sharpe": best_sharpe,
}, name="Fixed expert"))

Selected by validation return    slope/2
Selected by validation Sharpe      vol/2
Name: Fixed expert, dtype: object

## Fit on V1, select on V2

Validation is split chronologically in half. The boundary return is excluded.
The router uses V1 only. Ridge selects regularization on V2 without refitting.

In [8]:
cut = len(frames["val"]) // 2
v1 = frames["val"].iloc[:cut].reset_index(drop=True)
v2 = frames["val"].iloc[cut:].reset_index(drop=True)
d1, d2 = d_val[:cut - 1], d_val[cut:]
sigma = d1.astype(np.float64).reshape(-1, 6).std(axis=0, ddof=0)
assert (sigma > 0).all()

indicators = ["slope_360", "vol_360"]
router_bounds = np.quantile(v1[indicators], [0.35, 0.65], axis=0)


def market_states(frame):
    values = frame[indicators].iloc[:-1].to_numpy()
    levels = np.where(values < router_bounds[0], 0, np.where(values > router_bounds[1], 2, 1))
    return 3 * levels[:, 0] + levels[:, 1]


v1_results = {
    expert: account(v1, result["actions"][:len(v1) - 1])
    for expert, result in validation.items()
}
states = market_states(v1)
counts = np.bincount(states, minlength=9)
scores = pd.DataFrame({
    expert: result["step_pnl"] for expert, result in v1_results.items()
}).groupby(states).mean().reindex(range(9))
fallback = np.argmax([result["metrics"]["MTM Return (%)"] for result in v1_results.values()])
routing = np.array([
    scores.iloc[state].to_numpy().argmax() if counts[state] >= 100 else fallback
    for state in range(9)
])


def ridge_inputs(frame, advantages, contextual):
    values = advantages.astype(np.float64)
    if contextual:
        trend = frame.slope_360.to_numpy()[:-1, None, None]
        volatility = frame.vol_360.to_numpy()[:-1, None, None]
        values = np.concatenate([values, values * trend, values * volatility], axis=2)
    return values.reshape(len(values) * 2, -1)


target = np.repeat(v1.close.to_numpy()[1:] / v1.close.to_numpy()[:-1] - 1, 2)
ridge_models = {}
tuning = []
for name, contextual in (("Ridge-Static", False), ("Ridge-Contextual", True)):
    scaler = StandardScaler(with_mean=False)
    x1 = scaler.fit_transform(ridge_inputs(v1, d1, contextual))
    x2 = scaler.transform(ridge_inputs(v2, d2, contextual))
    best_tune_return = -np.inf
    for alpha in (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0):
        model = Ridge(alpha=alpha, fit_intercept=True, solver="cholesky")
        model.fit(x1, target, sample_weight=np.full(len(target), 0.5))
        forecast = model.predict(x2).reshape(len(d2), 2)
        tune_return = evaluate(v2, forecast, cost=FEE)["metrics"]["MTM Return (%)"]
        if tune_return > best_tune_return:
            best_tune_return = tune_return
            ridge_models[name] = (scaler, model, contextual)
    tuning.append({"Method": name, "Selected alpha": ridge_models[name][1].alpha})
display(pd.DataFrame(tuning).set_index("Method"))

,Selected alpha
Method,
Ridge-Static,1000.0
Ridge-Contextual,1000.0


## Review Table 2: aggregation on ETH test

In [9]:
d_test = expert_advantages("test")
test_frame = frames["test"]
individual = {
    expert: evaluate(test_frame, d_test[:, :, i])
    for i, expert in enumerate(EXPERTS)
}
test_results = {
    "Uniform Q": evaluate(test_frame, d_test.astype(np.float64).sum(axis=2)),
    "Normalized Q": evaluate(test_frame, (d_test.astype(np.float64) / sigma).sum(axis=2)),
    "Majority": evaluate(test_frame, (d_test > 0).sum(axis=2) - 3, keep_ties=True),
}
routes = routing[market_states(test_frame)]
test_results["Tabular hard router"] = evaluate(
    test_frame, d_test[np.arange(len(d_test)), :, routes]
)
for name, (scaler, model, contextual) in ridge_models.items():
    x = scaler.transform(ridge_inputs(test_frame, d_test, contextual))
    forecast = model.predict(x).reshape(len(d_test), 2)
    test_results[f"{name}, V1 -> V2"] = evaluate(test_frame, forecast, cost=FEE)
test_results["Fixed expert: validation return"] = individual[best_return]
test_results["Fixed expert: validation Sharpe"] = individual[best_sharpe]

aggregation = pd.DataFrame({name: result["metrics"] for name, result in test_results.items()}).T
display(aggregation.round({column: 3 if column == "Sharpe" else 2 for column in aggregation}))

,MTM Return (%),Author-style Return (%),Sharpe,MDD (%),Exposure (%)
Uniform Q,5.72,5.14,0.773,12.92,29.64
Normalized Q,-4.04,-3.53,-0.333,18.07,34.31
Majority,9.08,9.31,1.128,9.65,35.38
Tabular hard router,10.04,9.94,0.915,18.17,82.93
"Ridge-Static, V1 -> V2",13.36,13.22,1.414,12.96,46.80
"Ridge-Contextual, V1 -> V2",6.88,7.22,0.992,9.37,24.67
Fixed expert: validation return,23.28,23.87,2.424,9.54,32.09
Fixed expert: validation Sharpe,-22.47,-18.74,-3.116,25.28,22.80


## Review Table 1: trend-expert exposure

Regimes use retrospective 4,320-minute raw-price segments and train-derived cutoffs.
These labels are used only for this analysis.

In [10]:
def segment_slopes(raw):
    slopes = []
    for start in range(0, len(raw) - 4320 + 1, 4320):
        price = raw.close.iloc[start:start + 4320].to_numpy()
        slopes.append(preprocessing["get_slope"](preprocessing["smooth_data"](price)))
    return np.asarray(slopes)


train_cutoffs = np.quantile(segment_slopes(raw_prices["train"]), [0.35, 0.65])
# Merging the original outer bins leaves these two trend boundaries.
segment_regimes = pd.cut(
    segment_slopes(raw_prices["test"]),
    bins=[-np.inf, *train_cutoffs, np.inf],
    labels=["Bear", "Neutral", "Bull"],
)
raw_rows = pd.Index(raw_prices["test"].timestamp).get_indexer(test_frame.timestamp.iloc[:-1])
assert (raw_rows >= 0).all()
regimes = np.asarray(segment_regimes)[raw_rows // 4320]
trend_actions = np.column_stack([individual[name]["actions"] for name in EXPERTS[:3]])
exposure = pd.DataFrame(
    {regime: 100 * trend_actions[regimes == regime].mean(axis=0)
     for regime in ("Bear", "Neutral", "Bull")},
    index=pd.Index(EXPERTS[:3], name="Expert"),
)
display(exposure.round(2))
display(validation_metrics.loc[["slope/1", "slope/3"], ["Exposure (%)"]].round(2))

,Bear,Neutral,Bull
Expert,,,
slope/1,9.66,5.14,11.04
slope/2,26.44,26.71,49.96
slope/3,99.88,99.73,99.64


,Exposure (%)
slope/1,5.84
slope/3,99.56


## Exposure component of specialization

For each expert, the contrast is its PnL relative to the other two trend experts in its own regime minus the minute-weighted contrast in the other regimes.

In [11]:
price_changes = np.diff(test_frame.close.to_numpy())
net_pnl = np.column_stack([individual[name]["step_pnl"] for name in EXPERTS[:3]])
regime_names = ("Bear", "Neutral", "Bull")
regime_counts = np.array([(regimes == name).sum() for name in regime_names])
net_means = np.array([net_pnl[regimes == name].mean(axis=0) for name in regime_names])
exposure_means = np.array([
    HOLDING * trend_actions[regimes == name].mean(axis=0) * price_changes[regimes == name].mean()
    for name in regime_names
])


def specialization_contrast(means):
    relative = means - (means.sum(axis=1, keepdims=True) - means) / 2
    contrasts = []
    for i in range(3):
        other = np.arange(3) != i
        outside = np.average(relative[other, i], weights=regime_counts[other])
        contrasts.append(relative[i, i] - outside)
    return np.asarray(contrasts)


net_contrast = specialization_contrast(net_means)
exposure_contrast = specialization_contrast(exposure_means)
decomposition = pd.DataFrame({
    "Net contrast (mUSDT/min)": 1000 * net_contrast,
    "Exposure component (mUSDT/min)": 1000 * exposure_contrast,
    "Residual (mUSDT/min)": 1000 * (net_contrast - exposure_contrast),
    "Exposure share (%)": 100 * exposure_contrast / net_contrast,
}, index=EXPERTS[:3])
display(decomposition.loc[["slope/1", "slope/3"]].round(3))

,Net contrast (mUSDT/min),Exposure component (mUSDT/min),Residual (mUSDT/min),Exposure share (%)
slope/1,2.897,2.464,0.433,85.065
slope/3,3.767,3.263,0.504,86.618
